# Ruri v3 + LogisticRegression（実データ）

日本語Amazonレビューの2値分類データ [JGLUE/MARC-ja](https://huggingface.co/datasets/shunk031/JGLUE) を読み込み、[Ruri v3 310m](https://huggingface.co/cl-nagoya/ruri-v3-310m) の固定embeddingでpositive/negativeを分類する。

In [8]:
#%pip install -q -U "transformers==4.57.6" sentence-transformers datasets scikit-learn

In [9]:
import torch
#import torch._dynamo
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [10]:
base_url = (
    "https://huggingface.co/datasets/shunk031/JGLUE/"
    "resolve/refs%2Fconvert%2Fparquet/MARC-ja"
)
dataset = load_dataset(
    "parquet",
    data_files={
        "train": f"{base_url}/jglue-train.parquet",
        "validation": f"{base_url}/jglue-validation.parquet",
    },
)
print(dataset)
print(dataset["train"].features)
#print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'review_id'],
        num_rows: 187528
    })
    validation: Dataset({
        features: ['sentence', 'label', 'review_id'],
        num_rows: 5654
    })
})
{'sentence': Value('string'), 'label': ClassLabel(names=['positive', 'negative', 'neutral']), 'review_id': Value('string')}


In [11]:
# Colab向けの試行件数。全件使う場合は各splitをそのまま使用する。
N_TRAIN, N_VALID = 10_000, 2_000
train_ds = dataset["train"].shuffle(seed=42).select(range(N_TRAIN))
valid_ds = dataset["validation"].shuffle(seed=42).select(range(N_VALID))

train_texts = ["トピック: " + x for x in train_ds["sentence"]]
valid_texts = ["トピック: " + x for x in valid_ds["sentence"]]
y_train, y_valid = train_ds["label"], valid_ds["label"]

In [12]:
encoder = SentenceTransformer("cl-nagoya/ruri-v3-30m", device=device)
encoder.max_seq_length = 512  # レビュー処理の計算量を制限

X_train = encoder.encode(
    train_texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True
)
X_valid = encoder.encode(
    valid_texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True
)
print(X_train.shape, X_valid.shape)

Loading weights:   0%|          | 0/62 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(10000, 256) (2000, 256)


In [13]:
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train, y_train)

pred = classifier.predict(X_valid)
print("accuracy:", accuracy_score(y_valid, pred))
print("confusion matrix:\n", confusion_matrix(y_valid, pred))
print(classification_report(y_valid, pred, zero_division=0))

accuracy: 0.9345
confusion matrix:
 [[1683    3]
 [ 128  186]]
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1686
           1       0.98      0.59      0.74       314

    accuracy                           0.93      2000
   macro avg       0.96      0.80      0.85      2000
weighted avg       0.94      0.93      0.93      2000



In [14]:
def predict(text):
    embedding = encoder.encode(
        ["トピック: " + text], normalize_embeddings=True
    )
    label = int(classifier.predict(embedding)[0])
    probabilities = classifier.predict_proba(embedding)[0]
    return {
        "label": label,
        "probabilities": {int(c): float(p) for c, p in zip(classifier.classes_, probabilities)},
    }

predict("軽くて使いやすく、購入してよかったです。")

{'label': 0, 'probabilities': {0: 0.9956098946242323, 1: 0.004390105375767638}}